# LeetCode #1036: Escape a Large Maze

https://leetcode.com/problems/escape-a-large-maze/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(10^{12})$ | $O(10^{12})$ |
| **Optimal: Bounded BFS ★** | $O(b^2)$ | $O(b^2)$ |

---

## Understanding the Methods

### Brute Force
Run a full BFS on the $10^6 \times 10^6$ grid. This is impossibly large — we cannot allocate or visit that many cells.

### Optimal: Bounded BFS ★
$b$ blocked cells can enclose at most $\frac{b^2}{2}$ area (triangle). BFS from `source` with a cap of $b^2$ visits: if we exceed the cap without reaching `target`, `source` is not enclosed. Apply the same test from `target`. If neither is enclosed, escape is always possible.

**Constraints:**
* $1 \le blocked.length \le 200$
* Grid size $10^6 \times 10^6$

## Solutions

### C#

In [ ]:
using System.Collections.Generic;

public class Solution {
    static readonly int[][] Dirs = {{0,1},{0,-1},{1,0},{-1,0}};
    const int N = 1_000_000;

    public bool IsEscapePossible(int[][] blocked, int[] source, int[] target) {
        if (blocked.Length == 0) return true;
        var blockSet = new HashSet<long>();
        foreach (var b in blocked) blockSet.Add(Encode(b[0], b[1]));

        // Max area a b-cell barrier can enclose is b*(b-1)/2
        int maxCells = blocked.Length * blocked.Length / 2;
        // source must not be enclosed AND target must not be enclosed
        return Bfs(source, target, blockSet, maxCells) &&
               Bfs(target, source, blockSet, maxCells);
    }

    // Returns true if 'start' can reach 'end' or escape the enclosed region
    bool Bfs(int[] start, int[] end, HashSet<long> blockSet, int maxCells) {
        var visited = new HashSet<long>();
        var q = new Queue<int[]>();
        q.Enqueue(start);
        visited.Add(Encode(start[0], start[1]));

        while (q.Count > 0) {
            var cur = q.Dequeue();
            // Reached the target — definitely not enclosed
            if (cur[0] == end[0] && cur[1] == end[1]) return true;

            foreach (var d in Dirs) {
                int nr = cur[0] + d[0], nc = cur[1] + d[1];
                if (nr < 0 || nr >= N || nc < 0 || nc >= N) continue;
                long key = Encode(nr, nc);
                if (blockSet.Contains(key) || !visited.Add(key)) continue;
                q.Enqueue(new[]{nr, nc});
            }
            // Explored more cells than any barrier can enclose — open grid
            if (visited.Count > maxCells) return true;
        }
        return false; // BFS exhausted within cap — start is enclosed
    }

    long Encode(int r, int c) => (long)r * 1_000_000 + c;
}

### Python

In [ ]:
from collections import deque

class Solution:
    def isEscapePossible(self, blocked: list[list[int]], source: list[int], target: list[int]) -> bool:
        if not blocked:
            return True

        N = 1_000_000
        block_set = {(b[0], b[1]) for b in blocked}
        # b blocked cells can form a triangle enclosing at most b*(b-1)//2 cells
        max_cells = len(blocked) ** 2 // 2

        def bfs(start, end):
            sr, sc = start
            er, ec = end
            visited = {(sr, sc)}
            q = deque([(sr, sc)])
            while q:
                r, c = q.popleft()
                if (r, c) == (er, ec):
                    return True  # reached the other point — not enclosed
                for dr, dc in ((-1,0),(1,0),(0,-1),(0,1)):
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < N and 0 <= nc < N and (nr, nc) not in block_set and (nr, nc) not in visited:
                        visited.add((nr, nc))
                        q.append((nr, nc))
                if len(visited) > max_cells:
                    return True  # explored past the max enclosable area — open
            return False  # exhausted BFS inside enclosure

        return bfs(source, target) and bfs(target, source)

### Go

In [ ]:
func isEscapePossible(blocked [][]int, source []int, target []int) bool {
	const N = 1_000_000
	if len(blocked) == 0 {
		return true
	}
	blockSet := map[[2]int]bool{}
	for _, b := range blocked {
		blockSet[[2]int{b[0], b[1]}] = true
	}
	// Triangle bound: b cells can enclose at most b*b/2 area
	maxCells := len(blocked) * len(blocked) / 2

	type pt = [2]int
	dirs := []pt{{0,1},{0,-1},{1,0},{-1,0}}

	bfs := func(start, end pt) bool {
		visited := map[pt]bool{start: true}
		q := []pt{start}
		for len(q) > 0 {
			cur := q[0]; q = q[1:]
			if cur == end { return true }
			for _, d := range dirs {
				nr, nc := cur[0]+d[0], cur[1]+d[1]
				if nr < 0 || nr >= N || nc < 0 || nc >= N { continue }
				np := pt{nr, nc}
				if blockSet[np] || visited[np] { continue }
				visited[np] = true
				q = append(q, np)
			}
			if len(visited) > maxCells { return true }
		}
		return false
	}

	s, t := pt{source[0], source[1]}, pt{target[0], target[1]}
	return bfs(s, t) && bfs(t, s)
}

### Rust

In [ ]:
use std::collections::{HashSet, VecDeque};

impl Solution {
    pub fn is_escape_possible(blocked: Vec<Vec<i32>>, source: Vec<i32>, target: Vec<i32>) -> bool {
        const N: i32 = 1_000_000;
        if blocked.is_empty() { return true; }

        let block_set: HashSet<(i32,i32)> = blocked.iter().map(|b| (b[0], b[1])).collect();
        let max_cells = (blocked.len() * blocked.len() / 2) as usize;

        let bfs = |start: (i32,i32), end: (i32,i32)| -> bool {
            let mut visited = HashSet::new();
            visited.insert(start);
            let mut q = VecDeque::from([start]);
            while let Some((r, c)) = q.pop_front() {
                if (r, c) == end { return true; }
                for (dr, dc) in [(-1,0),(1,0),(0,-1),(0,1)] {
                    let (nr, nc) = (r + dr, c + dc);
                    if nr < 0 || nr >= N || nc < 0 || nc >= N { continue; }
                    if block_set.contains(&(nr, nc)) { continue; }
                    if visited.insert((nr, nc)) { q.push_back((nr, nc)); }
                }
                // Exceeded maximum enclosable area — source can escape
                if visited.len() > max_cells { return true; }
            }
            false
        };

        let s = (source[0], source[1]);
        let t = (target[0], target[1]);
        bfs(s, t) && bfs(t, s)
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `blocked=[], source=[0,0], target=[999999,999999]`
No blocked cells means no enclosure is possible, so the answer is immediately `true` — the early-exit guard fires.

### 2. Slightly Complex
**Input:** `blocked=[[0,1],[1,0]], source=[0,0], target=[999999,999999]`
Two cells can enclose at most 1 area cell ($2^2/2 = 2$). BFS from (0,0) reaches (0,0) alone — the cap is 2 but BFS exhausts after 1 cell (blocked on two sides). BFS from target reaches well beyond 2 cells, so target is not enclosed. Source is enclosed → returns `false`.

### 3. Edge Case: Time Factor
**Input:** `blocked` has 200 cells forming the tightest possible triangle; source is just inside.
BFS cap = $200^2/2 = 20000$. BFS visits exactly 20 000 cells without escaping → source declared enclosed. Total work: $O(b^2) = O(40000)$.

### 4. Edge Case: Space Factor
**Input:** `blocked` has 200 cells; neither source nor target is enclosed.
Both BFS runs each visit up to $\approx 20000$ cells before exceeding the cap, using $O(b^2)$ memory in the visited sets.

### 5. Almost-Impossible but Plausible
**Input:** `blocked` forms a half-arc of 200 cells; source is at the open end, target is far outside.
BFS from source quickly exits the half-arc and exceeds cap → `true`. BFS from target never approaches the arc → `true`. Result: escape is possible despite the intimidating barrier shape.